In [ ]:
# Cell 1 — self-contained bootstrap
from google.colab import userdata
import torch, os, sys

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
GITHUB_EMAIL = "evenjlinekka@gmail.com"
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Evenjlin/deep-space-interference-ml.git"

if not os.path.exists('/content/deep-space-interference-ml'):
    !git clone {REPO_URL} /content/deep-space-interference-ml
%cd /content/deep-space-interference-ml
!git config --global user.email "{GITHUB_EMAIL}"
!git config --global user.name "Evenjlin"
!git remote set-url origin {REPO_URL}
!git pull

!pip install -r requirements.txt -q
sys.path.insert(0, os.getcwd())

In [ ]:
# Cell 2 — config + data
import numpy as np
from sklearn.metrics import roc_auc_score
from src.channel import SignalConfig
from src.data_loader import make_clean_sample, build_test_set
from src.model import MLPAutoencoder, ae_score, vectorize
from src.train import debug_ladder, train_autoencoder, vectors_from_samples

cfg = SignalConfig(fd_max=0.0)  # BASE-PAPER FACT: Fig.3 condition, no freq shift
rng = np.random.default_rng(42)

N_SYMBOLS = 64   # OUR ASSUMPTION: matches an already-characterized Step 3 setting
SIR_DB, SNR_DB = 20.0, 15.0  # BASE-PAPER FACT (Fig.3 caption)
INPUT_DIM = 2 * N_SYMBOLS * cfg.NSPS
print("Input dim:", INPUT_DIM)

In [ ]:
# Cell 3 — build train/val pools (interference-free only, per BASE-PAPER FACT)
M_TRAIN, M_VAL = 4000, 1000
train_samples = [make_clean_sample(cfg, N_SYMBOLS, SNR_DB, rng) for _ in range(M_TRAIN)]
val_samples   = [make_clean_sample(cfg, N_SYMBOLS, SNR_DB, rng) for _ in range(M_VAL)]

train_vecs = vectors_from_samples(train_samples)
val_vecs   = vectors_from_samples(val_samples)
print("Train shape:", train_vecs.shape, " Val shape:", val_vecs.shape)

In [ ]:
# Cell 4 — DEBUG LADDER (mandatory before full training, per master prompt §15)
model = MLPAutoencoder(INPUT_DIM)
debug_ladder(model, train_vecs, device)

In [ ]:
# Cell 5 — full training (fresh model, debug ladder above already mutated its weights)
model = MLPAutoencoder(INPUT_DIM)
model, history = train_autoencoder(model, train_vecs, val_vecs, device, 
                                     epochs=50, batch_size=64, lr=1e-3, patience=5)

import matplotlib.pyplot as plt
plt.figure(figsize=(6,4))
plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("MSE loss"); plt.legend()
plt.title("Autoencoder training curve")
plt.tight_layout()
plt.savefig("figures/ae_training_curve.png", dpi=150)
plt.show()

os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/ae_detector_n64.pt")

In [ ]:
# Cell 6 — evaluate AE, compare to paper's Table 1 AE row and to our Step 3 PCA/Mahalanobis
paper_ae = {"tone": 0.7544, "chirp": 0.7816, "fawgn": 0.7615}  # BASE-PAPER FACT (Table 1)

ae_results = {}
for itype in ["tone", "chirp", "fawgn"]:
    samples, labels = build_test_set(cfg, N_SYMBOLS, rng, itype, 200, SIR_DB, SNR_DB)
    scores = [ae_score(x, model, device) for x in samples]
    auc = roc_auc_score(labels, scores)
    ae_results[itype] = auc
    print(f"{itype:6s}  AE AUC={auc:.4f}   paper AE AUC={paper_ae[itype]:.4f}")

import pandas as pd
comparison = pd.DataFrame({
    "paper_ae_auc": paper_ae,
    "our_ae_auc": ae_results,
})
comparison["difference"] = comparison["our_ae_auc"] - comparison["paper_ae_auc"]
print(comparison)
comparison.to_csv("results/ae_reproducibility_table.csv")